# 03. Model Development & Cross-Validation Benchmarking

**Author**: Nishshanka A. D. N. N. (IT24104095)  
**Role & Responsibility**: Model Development (Phase 3)  
**Module**: IT3091 Machine Learning  
**Project**: Campaign Response Prediction for Bank Marketing (Group ID: 2026-DS-12)  
**Ratified Governance**: ADR-001 (Stratified Random Holdout), ADR-002 (OHE Education), ADR-003 (Deduplication), ADR-004 (Dual Age Representation)  
**Constitution Alignment**: Principles I (Reproducibility), II (Pipeline-First), III (No Data Leakage), IV (Metric Transparency), V (Notebook Discipline), VI (Document As You Build)

---

### Notebook Purpose & Analytical Goals
1. Ingest preprocessed development features (`X_train_processed.csv`, `y_train.csv`, $N=32,940$, 71 features) and strictly verify the absence of `duration` (Data Leakage Prohibition).
2. Execute a leakage-free Stratified 5-Fold Cross-Validation protocol (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`).
3. Evaluate a portfolio of five diverse candidate architectures:
   - **Model 0 (Baseline)**: Majority / Prior No-Skill Baseline (`DummyClassifier`)
   - **Model 1 (Linear Benchmark)**: Cost-Sensitive L2-Regularized `LogisticRegression`
   - **Model 2 (Bagging Ensemble)**: Cost-Sensitive `RandomForestClassifier`
   - **Model 3 (Gradient Boosting)**: Cost-Sensitive `HistGradientBoostingClassifier`
   - **Model 4 (Margin / Neural)**: Multi-Layer Perceptron `MLPClassifier`
4. Optimize hyperparameters on top non-linear candidates via 5-fold CV on `X_train` to maximize PR-AUC (`average_precision`).
5. Synthesize comparative performance tables, distribution boxplots, and serialize model bundles to `models/` for downstream threshold optimization (Phase 4).


## Cell 1: Environment Setup & Deterministic Configuration
Initialize environment paths, import core modeling packages, and pin all random seeds (`random_state=42`) per Constitution Principle I.


In [1]:
import os
import sys
import json
import time
from pathlib import Path

# Resolve project root dynamically
current_dir = Path.cwd().resolve()
if (current_dir / "src" / "modeling.py").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src" / "modeling.py").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = next(
        (p for p in [current_dir] + list(current_dir.parents) if (p / "src" / "modeling.py").exists()),
        current_dir
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure MPLCONFIGDIR is set cleanly
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl_config")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV

from src.modeling import (
    evaluate_cv_model,
    compile_cv_benchmark_table,
    export_candidate_bundle
)

# Set global styles and seeds
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.autolayout": True,
    "font.sans-serif": "Arial",
    "axes.edgecolor": "#cccccc",
    "axes.linewidth": 0.8
})
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print(f"Project root resolved: {PROJECT_ROOT}")
print(f"NumPy version:        {np.__version__}")
print(f"Pandas version:       {pd.__version__}")
print(f"Scikit-Learn version: {sklearn.__version__}")
print(f"Joblib version:       {joblib.__version__}")
print(f"Random seed locked:   {RANDOM_STATE}")
print("Environment initialized successfully.")


Project root resolved: /Users/moni/Desktop/ML proj
NumPy version:        2.5.3
Pandas version:       3.0.6
Scikit-Learn version: 1.9.1
Joblib version:       1.6.0
Random seed locked:   42
Environment initialized successfully.


mkdir -p failed for path /private/tmp/mpl_config: [Errno 1] Operation not permitted: '/private/tmp/mpl_config'
Matplotlib created a temporary cache directory at /var/folders/21/5_d9lvkx55j2xppyx13xbhjh0000gn/T/matplotlib-lsuhenwz because there was an issue with MPLCONFIGDIR (/private/tmp/mpl_config); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


## Cell 2: Data Ingestion & Precondition Leakage Audit
Load preprocessed training data (`X_train_processed.csv`, `y_train.csv`) and validate dimensionality and zero-leakage constraints against `data/processed/feature_metadata.json`.


In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
metadata_path = DATA_DIR / "feature_metadata.json"

with open(metadata_path, "r", encoding="utf-8") as f:
    feature_meta = json.load(f)

X_train = pd.read_csv(DATA_DIR / "X_train_processed.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").iloc[:, 0]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Observed positive rate: {y_train.mean():.5f} (Expected: ~0.1127)")

# Constitutional and Precondition Assertions
assert X_train.shape == (32940, 71), f"Unexpected X_train shape: {X_train.shape}"
assert len(y_train) == 32940, f"Unexpected y_train length: {len(y_train)}"
assert "duration" not in X_train.columns, "CRITICAL CONSTITUTION VIOLATION: duration feature leaked into training data!"
assert X_train.isna().sum().sum() == 0, "Missing values detected in preprocessed training feature matrix!"
assert set(X_train.columns) == set(feature_meta["features_list"]), "Columns mismatch between dataset and feature_metadata.json!"

print("\n[LEAKAGE AUDIT PASS]: 'duration' is confirmed ABSENT from features.")
print("[SPLIT AUDIT PASS]: Test partition ('X_test_processed.csv') remains untouched and unread.")
print("[DATA INTEGRITY PASS]: All 32,940 training records and 71 features verified.")


X_train shape: (32940, 71)
y_train shape: (32940,)
Observed positive rate: 0.11266 (Expected: ~0.1127)

[LEAKAGE AUDIT PASS]: 'duration' is confirmed ABSENT from features.
[SPLIT AUDIT PASS]: Test partition ('X_test_processed.csv') remains untouched and unread.
[DATA INTEGRITY PASS]: All 32,940 training records and 71 features verified.


## Cell 3: Stratified 5-Fold Cross-Validation Harness & Metric Suite
Initialize the `StratifiedKFold` protocol and registry for fold-level metrics:
- **Primary Metric**: PR-AUC (`average_precision`) to evaluate precision-recall trade-offs under class skew without true-negative inflation.
- **Diagnostic Metrics**: ROC-AUC, Balanced Accuracy, F1-macro, F1-minority, Brier Score Loss, and Train-Val Overfitting Gap.


In [3]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Cross-Validation Setup: StratifiedKFold(n_splits=5, shuffle=True, random_state={RANDOM_STATE})")

# Track all candidate fold results and trained models
cv_fold_results = []
trained_models = {}

pos_counts = [y_train.iloc[val_idx].sum() for _, val_idx in cv_strategy.split(X_train, y_train)]
print(f"Minority class instances per validation fold: {pos_counts} (Mean: {np.mean(pos_counts):.1f})")


Cross-Validation Setup: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Minority class instances per validation fold: [np.int64(742), np.int64(742), np.int64(742), np.int64(742), np.int64(743)] (Mean: 742.2)


## Cell 4: Model 0 — Baseline (No-Skill / Prior Class Ratio)
Establish the theoretical performance floor using `DummyClassifier(strategy='prior')`.


In [4]:
dummy_clf = DummyClassifier(strategy="prior")

fold_df_dummy, summary_dummy = evaluate_cv_model(
    estimator=dummy_clf,
    X=X_train,
    y=y_train,
    model_name="Dummy Baseline",
    family="Baseline",
    cv=cv_strategy,
    random_state=RANDOM_STATE
)

cv_fold_results.append(fold_df_dummy)
dummy_clf.fit(X_train, y_train)
trained_models["Dummy Baseline"] = dummy_clf

print("--- Model 0 (Dummy Baseline) Cross-Validation Results ---")
print(f"Validation PR-AUC:       {summary_dummy['val_pr_auc_mean']:.4f} ± {summary_dummy['val_pr_auc_std']:.4f} (Theoretical floor: {y_train.mean():.4f})")
print(f"Validation ROC-AUC:      {summary_dummy['val_roc_auc_mean']:.4f} ± {summary_dummy['val_roc_auc_std']:.4f}")
print(f"Validation Balanced Acc: {summary_dummy['val_balanced_acc_mean']:.4f}")
print(f"Validation Brier Score:  {summary_dummy['val_brier_score_mean']:.4f}")


--- Model 0 (Dummy Baseline) Cross-Validation Results ---
Validation PR-AUC:       0.1127 ± 0.0001 (Theoretical floor: 0.1127)
Validation ROC-AUC:      0.5000 ± 0.0000
Validation Balanced Acc: 0.5000
Validation Brier Score:  0.1000


## Cell 5: Model 1 — Linear Benchmark (Cost-Sensitive Logistic Regression)
Evaluate L2-regularized `LogisticRegression` with `class_weight='balanced'` to establish the standard linear baseline.


In [5]:
logreg_clf = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

fold_df_logreg, summary_logreg = evaluate_cv_model(
    estimator=logreg_clf,
    X=X_train,
    y=y_train,
    model_name="Logistic Regression (Balanced)",
    family="Linear",
    cv=cv_strategy,
    random_state=RANDOM_STATE
)

cv_fold_results.append(fold_df_logreg)
logreg_clf.fit(X_train, y_train)
trained_models["Logistic Regression"] = logreg_clf

print("--- Model 1 (Logistic Regression) Cross-Validation Results ---")
print(f"Validation PR-AUC:       {summary_logreg['val_pr_auc_mean']:.4f} ± {summary_logreg['val_pr_auc_std']:.4f}")
print(f"Validation ROC-AUC:      {summary_logreg['val_roc_auc_mean']:.4f} ± {summary_logreg['val_roc_auc_std']:.4f}")
print(f"Validation Balanced Acc: {summary_logreg['val_balanced_acc_mean']:.4f}")
print(f"Validation F1-Minority:  {summary_logreg['val_f1_minority_mean']:.4f}")
print(f"Validation Brier Score:  {summary_logreg['val_brier_score_mean']:.4f}")
print(f"Train-Val PR-AUC Gap:    {summary_logreg['train_val_pr_auc_gap']:.4f} (Fit Time: {summary_logreg['fit_time_mean_sec']:.2f}s)")


--- Model 1 (Logistic Regression) Cross-Validation Results ---
Validation PR-AUC:       0.4454 ± 0.0128
Validation ROC-AUC:      0.7904 ± 0.0081
Validation Balanced Acc: 0.7402
Validation F1-Minority:  0.4524
Validation Brier Score:  0.1644
Train-Val PR-AUC Gap:    0.0068 (Fit Time: 0.10s)


## Cell 6: Model 2 — Bagging Ensemble (Random Forest Classifier)
Evaluate non-linear tree bagging with `RandomForestClassifier` utilizing balanced class weighting and regularization constraints.


In [6]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

fold_df_rf, summary_rf = evaluate_cv_model(
    estimator=rf_clf,
    X=X_train,
    y=y_train,
    model_name="Random Forest (Balanced)",
    family="Bagging Ensemble",
    cv=cv_strategy,
    random_state=RANDOM_STATE
)

cv_fold_results.append(fold_df_rf)
rf_clf.fit(X_train, y_train)
trained_models["Random Forest"] = rf_clf

print("--- Model 2 (Random Forest) Cross-Validation Results ---")
print(f"Validation PR-AUC:       {summary_rf['val_pr_auc_mean']:.4f} ± {summary_rf['val_pr_auc_std']:.4f}")
print(f"Validation ROC-AUC:      {summary_rf['val_roc_auc_mean']:.4f} ± {summary_rf['val_roc_auc_std']:.4f}")
print(f"Validation Balanced Acc: {summary_rf['val_balanced_acc_mean']:.4f}")
print(f"Validation F1-Minority:  {summary_rf['val_f1_minority_mean']:.4f}")
print(f"Validation Brier Score:  {summary_rf['val_brier_score_mean']:.4f}")
print(f"Train-Val PR-AUC Gap:    {summary_rf['train_val_pr_auc_gap']:.4f} (Fit Time: {summary_rf['fit_time_mean_sec']:.2f}s)")


--- Model 2 (Random Forest) Cross-Validation Results ---
Validation PR-AUC:       0.4664 ± 0.0156
Validation ROC-AUC:      0.7992 ± 0.0112
Validation Balanced Acc: 0.7458
Validation F1-Minority:  0.4796
Validation Brier Score:  0.1496
Train-Val PR-AUC Gap:    0.1328 (Fit Time: 0.46s)


## Cell 7: Model 3 — Gradient Boosting (HistGradientBoosting Classifier)
Evaluate histogram-based gradient boosting (`HistGradientBoostingClassifier`) with balanced weighting, natively optimizing loss gradients across the 71 one-hot features.


In [7]:
hgb_clf = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

fold_df_hgb, summary_hgb = evaluate_cv_model(
    estimator=hgb_clf,
    X=X_train,
    y=y_train,
    model_name="HistGradientBoosting (Balanced)",
    family="Gradient Boosting",
    cv=cv_strategy,
    random_state=RANDOM_STATE
)

cv_fold_results.append(fold_df_hgb)
hgb_clf.fit(X_train, y_train)
trained_models["HistGradientBoosting"] = hgb_clf

print("--- Model 3 (HistGradientBoosting) Cross-Validation Results ---")
print(f"Validation PR-AUC:       {summary_hgb['val_pr_auc_mean']:.4f} ± {summary_hgb['val_pr_auc_std']:.4f}")
print(f"Validation ROC-AUC:      {summary_hgb['val_roc_auc_mean']:.4f} ± {summary_hgb['val_roc_auc_std']:.4f}")
print(f"Validation Balanced Acc: {summary_hgb['val_balanced_acc_mean']:.4f}")
print(f"Validation F1-Minority:  {summary_hgb['val_f1_minority_mean']:.4f}")
print(f"Validation Brier Score:  {summary_hgb['val_brier_score_mean']:.4f}")
print(f"Train-Val PR-AUC Gap:    {summary_hgb['train_val_pr_auc_gap']:.4f} (Fit Time: {summary_hgb['fit_time_mean_sec']:.2f}s)")


--- Model 3 (HistGradientBoosting) Cross-Validation Results ---
Validation PR-AUC:       0.4662 ± 0.0137
Validation ROC-AUC:      0.7981 ± 0.0088
Validation Balanced Acc: 0.7469
Validation F1-Minority:  0.4736
Validation Brier Score:  0.1510
Train-Val PR-AUC Gap:    0.0686 (Fit Time: 1.20s)


## Cell 8: Model 4 — Alternative Architecture (Multi-Layer Perceptron MLP)
Evaluate a multi-layer perceptron neural network (`MLPClassifier`) with early stopping and L2 weight penalty.


In [8]:
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    alpha=0.01,
    early_stopping=True,
    n_iter_no_change=10,
    max_iter=200,
    random_state=RANDOM_STATE
)

fold_df_mlp, summary_mlp = evaluate_cv_model(
    estimator=mlp_clf,
    X=X_train,
    y=y_train,
    model_name="MLP Classifier (Neural)",
    family="Neural Network",
    cv=cv_strategy,
    random_state=RANDOM_STATE
)

cv_fold_results.append(fold_df_mlp)
mlp_clf.fit(X_train, y_train)
trained_models["MLP Classifier"] = mlp_clf

print("--- Model 4 (MLP Neural Network) Cross-Validation Results ---")
print(f"Validation PR-AUC:       {summary_mlp['val_pr_auc_mean']:.4f} ± {summary_mlp['val_pr_auc_std']:.4f}")
print(f"Validation ROC-AUC:      {summary_mlp['val_roc_auc_mean']:.4f} ± {summary_mlp['val_roc_auc_std']:.4f}")
print(f"Validation Balanced Acc: {summary_mlp['val_balanced_acc_mean']:.4f}")
print(f"Validation F1-Minority:  {summary_mlp['val_f1_minority_mean']:.4f}")
print(f"Validation Brier Score:  {summary_mlp['val_brier_score_mean']:.4f}")
print(f"Train-Val PR-AUC Gap:    {summary_mlp['train_val_pr_auc_gap']:.4f} (Fit Time: {summary_mlp['fit_time_mean_sec']:.2f}s)")


--- Model 4 (MLP Neural Network) Cross-Validation Results ---
Validation PR-AUC:       0.4461 ± 0.0173
Validation ROC-AUC:      0.7881 ± 0.0142
Validation Balanced Acc: 0.6070
Validation F1-Minority:  0.3379
Validation Brier Score:  0.0794
Train-Val PR-AUC Gap:    0.0453 (Fit Time: 1.21s)


## Cell 9: Structured Hyperparameter Optimization on Top Candidates
Execute systematic grid searches (`GridSearchCV`) with Stratified 5-Fold CV on `X_train` optimizing for `average_precision` (PR-AUC) on the top non-linear candidates:
1. `HistGradientBoostingClassifier`
2. `RandomForestClassifier`


In [9]:
print("=== HYPERPARAMETER OPTIMIZATION (5-FOLD CV ON X_TRAIN ONLY) ===")

# 1. Tuning HistGradientBoosting
hgb_grid = {
    "learning_rate": [0.04, 0.08],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [20, 50],
    "l2_regularization": [0.0, 1.0]
}

print("Searching HistGradientBoosting parameter grid (16 combinations x 5 folds = 80 fits)...")
t0 = time.time()
hgb_search = GridSearchCV(
    HistGradientBoostingClassifier(max_iter=150, class_weight="balanced", random_state=RANDOM_STATE),
    param_grid=hgb_grid,
    scoring="average_precision",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True
)
hgb_search.fit(X_train, y_train)
hgb_tune_time = time.time() - t0

print(f"HistGradientBoosting Best Params: {hgb_search.best_params_}")
print(f"Best Mean Validation PR-AUC:     {hgb_search.best_score_:.4f} (Search Time: {hgb_tune_time:.1f}s)")

# Evaluate tuned HGB across the standard metric suite
best_hgb = hgb_search.best_estimator_
fold_df_best_hgb, summary_best_hgb = evaluate_cv_model(
    estimator=best_hgb,
    X=X_train,
    y=y_train,
    model_name="HistGradientBoosting (Tuned)",
    family="Gradient Boosting",
    cv=cv_strategy,
    random_state=RANDOM_STATE
)
cv_fold_results.append(fold_df_best_hgb)
trained_models["HistGradientBoosting (Tuned)"] = best_hgb

# 2. Tuning Random Forest
rf_grid = {
    "n_estimators": [100, 150],
    "max_depth": [10, 14],
    "min_samples_leaf": [5, 10]
}

print("\nSearching RandomForest parameter grid (8 combinations x 5 folds = 40 fits)...")
t0 = time.time()
rf_search = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=rf_grid,
    scoring="average_precision",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True
)
rf_search.fit(X_train, y_train)
rf_tune_time = time.time() - t0

print(f"RandomForest Best Params:         {rf_search.best_params_}")
print(f"Best Mean Validation PR-AUC:     {rf_search.best_score_:.4f} (Search Time: {rf_tune_time:.1f}s)")

best_rf = rf_search.best_estimator_
fold_df_best_rf, summary_best_rf = evaluate_cv_model(
    estimator=best_rf,
    X=X_train,
    y=y_train,
    model_name="Random Forest (Tuned)",
    family="Bagging Ensemble",
    cv=cv_strategy,
    random_state=RANDOM_STATE
)
cv_fold_results.append(fold_df_best_rf)
trained_models["Random Forest (Tuned)"] = best_rf


=== HYPERPARAMETER OPTIMIZATION (5-FOLD CV ON X_TRAIN ONLY) ===
Searching HistGradientBoosting parameter grid (16 combinations x 5 folds = 80 fits)...
HistGradientBoosting Best Params: {'l2_regularization': 0.0, 'learning_rate': 0.04, 'max_leaf_nodes': 15, 'min_samples_leaf': 20}
Best Mean Validation PR-AUC:     0.4680 (Search Time: 46.5s)

Searching RandomForest parameter grid (8 combinations x 5 folds = 40 fits)...
RandomForest Best Params:         {'max_depth': 14, 'min_samples_leaf': 10, 'n_estimators': 150}
Best Mean Validation PR-AUC:     0.4668 (Search Time: 32.4s)


## Cell 10: Comparative Summary & Visual Cross-Validation Benchmarking
Synthesize all evaluated models into a unified benchmark comparison table and visualize fold score distributions.


In [10]:
# Compile ranked summary table
benchmark_summary_table = compile_cv_benchmark_table(cv_fold_results)
print("\n=== STRATIFIED 5-FOLD CV MODEL COMPARISON TABLE ===")
display_df = benchmark_summary_table.copy()
print(display_df.to_string(index=False))

# Visualization: PR-AUC and ROC-AUC Distributions Across 5 Folds
all_fold_df = pd.concat(cv_fold_results, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

# PR-AUC Plot
sns.boxplot(
    data=all_fold_df,
    x="val_pr_auc",
    y="model_name",
    palette="viridis",
    ax=axes[0],
    hue="model_name",
    legend=False
)
axes[0].set_title("5-Fold Cross-Validation PR-AUC (Primary Metric)", fontsize=12, fontweight="bold", pad=10)
axes[0].set_xlabel("Precision-Recall AUC (Average Precision)", fontsize=11)
axes[0].set_ylabel("")
axes[0].axvline(y_train.mean(), color="red", linestyle="--", alpha=0.7, label=f"No-Skill Baseline ({y_train.mean():.3f})")
axes[0].legend(loc="lower right")

# ROC-AUC Plot
sns.boxplot(
    data=all_fold_df,
    x="val_roc_auc",
    y="model_name",
    palette="mako",
    ax=axes[1],
    hue="model_name",
    legend=False
)
axes[1].set_title("5-Fold Cross-Validation ROC-AUC", fontsize=12, fontweight="bold", pad=10)
axes[1].set_xlabel("Receiver Operating Characteristic AUC", fontsize=11)
axes[1].set_ylabel("")
axes[1].axvline(0.5, color="red", linestyle="--", alpha=0.7, label="Random Guess (0.50)")
axes[1].legend(loc="lower right")

plt.suptitle("Candidate Model Portfolio Benchmark (Stratified 5-Fold CV on X_train)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()



=== STRATIFIED 5-FOLD CV MODEL COMPARISON TABLE ===
 Rank                           Model            Family Val PR-AUC (Mean ± Std) Val ROC-AUC (Mean ± Std) Val Balanced Acc Val F1-Minority Val Brier Score Train-Val Gap Fit Time (s)
    1    HistGradientBoosting (Tuned) Gradient Boosting         0.4680 ± 0.0148          0.8013 ± 0.0090           0.7464          0.4688          0.1560        0.0345         1.58
    2           Random Forest (Tuned)  Bagging Ensemble         0.4668 ± 0.0173          0.7989 ± 0.0109           0.7461          0.4764          0.1497        0.1131         0.74
    3        Random Forest (Balanced)  Bagging Ensemble         0.4664 ± 0.0156          0.7992 ± 0.0112           0.7458          0.4796          0.1496        0.1328         0.46
    4 HistGradientBoosting (Balanced) Gradient Boosting         0.4662 ± 0.0137          0.7981 ± 0.0088           0.7469          0.4736          0.1510        0.0686         1.20
    5         MLP Classifier (Neural)    N

## Cell 11: Model Artifact Persistence & Phase Handover
Serialize candidate models, grid search objects, and benchmark tables to `models/` for Phase 4 (Evaluation, Threshold Tuning & Decile Lift).


In [11]:
EXPORT_DIR = PROJECT_ROOT / "models"
os.makedirs(EXPORT_DIR, exist_ok=True)

export_metadata = {
    "project": "UCI Bank Marketing Campaign Response",
    "phase": "03_Model_Development",
    "author": "Nishshanka A. D. N. N. (IT24104095)",
    "date": "2026-10-10",
    "cv_strategy": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "primary_metric": "PR-AUC (average_precision)",
    "best_model_name": "HistGradientBoosting (Tuned)",
    "best_model_val_pr_auc": float(summary_best_hgb['val_pr_auc_mean']),
    "best_model_val_roc_auc": float(summary_best_hgb['val_roc_auc_mean']),
    "leakage_prevention": "duration strictly excluded; holdout test partition untouched"
}

models_payload = {
    "models": trained_models,
    "grid_searches": {
        "hist_gradient_boosting": hgb_search,
        "random_forest": rf_search
    },
    "fold_level_records": all_fold_df.to_dict(orient="records")
}

saved_paths = export_candidate_bundle(
    models_dict=models_payload,
    benchmark_table=benchmark_summary_table,
    export_dir=str(EXPORT_DIR),
    metadata=export_metadata
)

print(f"\n[ARTIFACT EXPORT SUCCESS]")
print(f"Serialized Bundle: {saved_paths['bundle_path']} ({os.path.getsize(saved_paths['bundle_path']) / (1024*1024):.2f} MB)")
print(f"CSV Benchmark:     {saved_paths['csv_path']}")
print("\nPhase 3 Model Development complete and ready for Phase 4 Evaluation & Diagnostics.")



[ARTIFACT EXPORT SUCCESS]
Serialized Bundle: /Users/moni/Desktop/ML proj/models/candidate_models.joblib (19.40 MB)
CSV Benchmark:     /Users/moni/Desktop/ML proj/models/cv_benchmark_summary.csv

Phase 3 Model Development complete and ready for Phase 4 Evaluation & Diagnostics.
